# Mid-update QC sweep — 2026 Q3 gas pipelines update

During-data-entry QC on the live "Pipelines (Gas/Oil/NGL) - main" sheet:
fuel/status/ProjectID sweeps (via `gem-tracker-constants`, the same buckets
the release pipeline filters on) plus data-entry checks focused on rows
touched this cycle. Run at least monthly, and after any bulk change.

**Read-only** — this notebook only reports; fix findings in the live sheet
by hand.

In [ ]:
%pip install -q -e ../../gem-tracker-constants

In [ ]:
import re

import pandas as pd
import pygsheets

from gem_tracker_constants import (
    GAS_FUEL_OPTIONS,
    PIPELINE_STATUS,
    find_uncovered_fuels,
)

pd.set_option('display.max_rows', 250)

## Config

In [ ]:
# live backend: "Pipelines (Gas/Oil/NGL) - main"
PIPELINES_SHEET_KEY = '1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek'
PIPELINES_TAB = 'Gas pipelines'

# rows with LastUpdated on/after this date get the extra data-entry checks
CYCLE_START = '2026-07-06'

# columns shown when displaying offending rows
ID_COLS = ['ProjectID', 'PipelineName', 'SegmentName', 'CountriesOrAreas',
           'Status', 'Fuel', 'Researcher', 'LastUpdated']

YEAR_COLS = ['ProposalYear', 'ConstructionYear', 'StartYear1', 'StartYearEarliest',
             'ShelvedYear', 'CancelledYear', 'StopYear']
YEAR_RANGE = (1900, 2050)

findings = {}  # check name -> offending-row count, summarized at the end

## Pull the live sheet

In [ ]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
ss = gc.open_by_key(PIPELINES_SHEET_KEY)
df = ss.worksheet('title', PIPELINES_TAB).get_as_df(start='A3', include_tailing_empty=False)
print(f'{len(df)} rows on "{PIPELINES_TAB}"')

## 1. Fuel sweep

Every `Fuel` value must fall in a `gem-tracker-constants` bucket — the
release downloads and QC summary sheets filter on those buckets, so anything
uncovered silently drops out of the release.

In [ ]:
uncovered = find_uncovered_fuels(df, fuel_col='Fuel')
findings['fuels outside constants buckets'] = len(uncovered)

## 2. Status sweep

In [ ]:
status_ok = df['Status'].isin(PIPELINE_STATUS)
fixable = (~status_ok) & df['Status'].astype(str).str.strip().str.lower().isin(PIPELINE_STATUS)

findings['status not in canonical list'] = int((~status_ok).sum())
print(f'canonical statuses: {PIPELINE_STATUS}')
print(f'{int((~status_ok).sum())} rows with non-canonical Status '
      f'({int(fixable.sum())} of them just case/whitespace)')
if (~status_ok).any():
    display(df.loc[~status_ok, ID_COLS].assign(
        Status_repr=df.loc[~status_ok, 'Status'].map(repr)))

## 3. ProjectID sweep

In [ ]:
pid = df['ProjectID'].astype(str).str.strip()

missing_pid = pid == ''
bad_format = ~missing_pid & ~pid.str.fullmatch(r'P\d+')
dupes = pid.duplicated(keep=False) & ~missing_pid

findings['missing ProjectID'] = int(missing_pid.sum())
findings['malformed ProjectID'] = int(bad_format.sum())
findings['duplicate ProjectID'] = int(dupes.sum())

for name, mask in [('missing', missing_pid), ('malformed (not P<digits>)', bad_format),
                   ('duplicated', dupes)]:
    print(f'{int(mask.sum())} rows with {name} ProjectID')
    if mask.any():
        display(df.loc[mask, ID_COLS].sort_values('ProjectID'))

## 4. Rows touched this cycle — data-entry checks

Missing required fields, unstamped researcher initials, and unparseable
`LastUpdated` dates on rows edited during this cycle.

In [ ]:
last_updated = pd.to_datetime(
    df['LastUpdated'].astype(str).str.strip(), errors='coerce', format='mixed')
has_date_text = df['LastUpdated'].astype(str).str.strip() != ''

unparseable_date = has_date_text & last_updated.isna()
findings['unparseable LastUpdated'] = int(unparseable_date.sum())
print(f'{int(unparseable_date.sum())} rows with unparseable LastUpdated')
if unparseable_date.any():
    display(df.loc[unparseable_date, ID_COLS])

recent = last_updated >= pd.Timestamp(CYCLE_START)
print(f'\n{int(recent.sum())} rows stamped on/after {CYCLE_START}')

required = ['PipelineName', 'Status', 'Fuel', 'CountriesOrAreas', 'Researcher']
for col in required:
    blank = recent & (df[col].astype(str).str.strip() == '')
    findings[f'cycle rows missing {col}'] = int(blank.sum())
    if blank.any():
        print(f'{int(blank.sum())} cycle rows missing {col}')
        display(df.loc[blank, ID_COLS])

cap_no_units = (recent
                & (df['Capacity'].astype(str).str.strip() != '')
                & (df['CapacityUnits'].astype(str).str.strip() == ''))
findings['cycle rows with Capacity but no CapacityUnits'] = int(cap_no_units.sum())
if cap_no_units.any():
    print(f'{int(cap_no_units.sum())} cycle rows with Capacity but no CapacityUnits')
    display(df.loc[cap_no_units, ID_COLS + ['Capacity', 'CapacityUnits']])

## 5. Year sanity

In [ ]:
for col in YEAR_COLS:
    if col not in df.columns:
        continue
    years = pd.to_numeric(df[col].astype(str).str.strip(), errors='coerce')
    has_text = df[col].astype(str).str.strip() != ''
    outliers = has_text & years.notna() & ~years.between(*YEAR_RANGE)
    non_numeric = has_text & years.isna()
    findings[f'{col} outside {YEAR_RANGE}'] = int(outliers.sum())
    if outliers.any():
        print(f'{int(outliers.sum())} rows with {col} outside {YEAR_RANGE}')
        display(df.loc[outliers, ID_COLS + [col]])
    if non_numeric.any():
        print(f'note: {int(non_numeric.sum())} rows with non-numeric {col} '
              '(free-text values, not flagged)')

## Summary

In [ ]:
print(f'QC sweep of "{PIPELINES_TAB}" ({len(df)} rows), cycle start {CYCLE_START}\n')
width = max(len(k) for k in findings)
clean = True
for check, n in findings.items():
    flag = 'ok' if n == 0 else f'ATTENTION ({n})'
    if n:
        clean = False
    print(f'  {check:<{width}}  {flag}')
print('\nall clean' if clean else '\nfix findings in the live sheet by hand — '
      'this notebook never writes to it')